# Pipeline conversacional de Chatbot com NLP (chatbot_novo)

- **Repositório:** `nlp-llms-portfolio`  
- **Tecnologias:** NLTK, spaCy, Scikit-Learn  
- **Arquivo Original:** `Chatbot_Novo.ipynb`  

---


## Bibliotecas

In [1]:
!pip install --upgrade pip
!pip install nltk
!pip install spacy
!python -m spacy download pt_core_news_sm
!pip install --upgrade numpy scipy gensim
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 67.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 138.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 147.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 46.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully

## Importação e Downloads

In [13]:
import nltk
import re
import string
import numpy as np
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import spacy
from gensim.models import Word2Vec
import multiprocessing

# Downloads de recursos do NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('rslp')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Package rslp is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Carregamento do livro

In [14]:
import requests

url = 'https://raw.githubusercontent.com/VictorSilvaCamargo/o-guarani/main/o-guarani.txt'
resposta = requests.get(url)

if resposta.status_code == 200:
    livro = resposta.text
    print(f"Livro carregado! Total de caracteres: {len(livro)}")
else:
    print("Erro ao carregar o livro:", resposta.status_code)


Livro carregado! Total de caracteres: 618788


## Pré Processamento

In [15]:
def preprocessar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'\s+', ' ', texto)
    texto = re.sub(r'\n', ' ', texto)
    texto = re.sub(r'[^\w\s]', '', texto)
    texto = re.sub(r"[ \t]+", " ", texto)

    tokens = word_tokenize(texto, language='portuguese')
    stop_words = set(stopwords.words('portuguese'))
    tokens_filtrados = [word for word in tokens if word not in stop_words and len(word) > 2]

    stemmer = RSLPStemmer()
    tokens_stemizados = [stemmer.stem(word) for word in tokens_filtrados]

    return tokens_stemizados

tokens_processados = preprocessar_texto(livro)
print(f"Total de tokens processados: {len(tokens_processados)}")

Total de tokens processados: 57085


## Pré Processamento para o Word2Vec

In [16]:
def preprocessar_texto_para_word2vec(texto):
    sentencas = sent_tokenize(texto, language='portuguese')
    stemmer = RSLPStemmer()
    stop_words = set(stopwords.words('portuguese'))
    sentencas_processadas = []

    for sent in sentencas:
        sent = sent.lower()
        sent = re.sub(r'\s+', ' ', sent)
        sent = re.sub(r'[^\w\s]', '', sent)
        tokens = word_tokenize(sent, language='portuguese')
        tokens_filtrados = [
            stemmer.stem(word)
            for word in tokens
            if word not in stop_words and len(word) > 2
        ]
        if len(tokens_filtrados) > 3:
            sentencas_processadas.append(tokens_filtrados)

    return sentencas_processadas

sentencas_processadas = preprocessar_texto_para_word2vec(livro)

## Treinamento Word2Vec

In [17]:
model_w2v = Word2Vec(
    sentences=sentencas_processadas,
    vector_size=100,
    window=5,
    sg=1,
    min_count=3,
    workers=multiprocessing.cpu_count(),
    epochs=15
)
model_w2v.save("word2vec_skipgram_o_guarani.model")

## Processamento Avançado

In [18]:
nlp = spacy.load("pt_core_news_sm")

def normalizar_entidade(ent):
    ent = ent.lower().strip()
    ent = re.sub(r"[^\w\s]", "", ent)
    ent = re.sub(r"\s+", " ", ent)
    return ent

def obter_vetor_medio(tokens, modelo_word2vec):
    vetores = []
    for word in tokens:
        if word in modelo_word2vec.wv:
            vetores.append(modelo_word2vec.wv[word])
    if vetores:
        return np.mean(vetores, axis=0)
    return None

def extrair_entidades_filtradas(texto):
    doc = nlp(texto)
    pessoas = {normalizar_entidade(ent.text) for ent in doc.ents if ent.label_ == "PER" and len(normalizar_entidade(ent.text).split()) >= 2}
    locais = {normalizar_entidade(ent.text) for ent in doc.ents if ent.label_ == "LOC" and len(normalizar_entidade(ent.text).split()) >= 2}
    return list(pessoas), list(locais)

## Criação de Chunks e Mapeamento

In [19]:
stemmer_global = RSLPStemmer()
stop_words_global = set(stopwords.words("portuguese"))

def preprocessar_sentenca(sent):
    sent = sent.lower()
    sent = re.sub(r"\s+", " ", sent)
    sent = re.sub(r"[^\w\s]", "", sent)
    tokens = word_tokenize(sent, language="portuguese")
    return [
        stemmer_global.stem(w)
        for w in tokens
        if (w not in stop_words_global and w.isalpha() and len(w) > 2)
    ]

sentencas_originais = sent_tokenize(livro, language="portuguese")

def criar_chunks(texto_completo, num_tokens_por_chunk=100, sobreposicao=20):

    sentencas = sent_tokenize(texto_completo, language='portuguese')
    chunks = []
    chunk_atual = []
    tokens_no_chunk = 0

    for sentenca in sentencas:
        tokens_sentenca = preprocessar_texto(sentenca)

        if tokens_no_chunk + len(tokens_sentenca) <= num_tokens_por_chunk:
            chunk_atual.append(sentenca)
            tokens_no_chunk += len(tokens_sentenca)
        else:
            if chunk_atual:
                chunks.append(" ".join(chunk_atual))

            # Inicia um novo chunk com sobreposição
            # Calcula o número de sentenças para a sobreposição de forma aproximada
            num_sentencas_sobreposicao = int(len(chunk_atual) * (sobreposicao / num_tokens_por_chunk))
            start_index_overlap = max(0, len(chunk_atual) - num_sentencas_sobreposicao)

            chunk_atual = chunk_atual[start_index_overlap:]
            tokens_no_chunk = sum(len(preprocessar_texto(s)) for s in chunk_atual)

            chunk_atual.append(sentenca)
            tokens_no_chunk += len(tokens_sentenca)

    if chunk_atual:
        chunks.append(" ".join(chunk_atual))

    return chunks

## Vetorização e Mapeamento

In [20]:
def vetor_medio_chunk(tokens, modelo):
    vetores = [modelo.wv[w] for w in tokens if w in modelo.wv]
    if vetores:
        return np.mean(vetores, axis=0)
    else:
        return np.zeros(modelo.vector_size)

mapa_chunks = []
for tokens, texto_chunk in chunks_legiveis:
    vetor = vetor_medio_chunk(tokens, model_w2v)
    entidades_chunk = extrair_entidades_filtradas(texto_chunk)
    mapa_chunks.append((vetor, texto_chunk, entidades_chunk))

## Sistema de Respostas e Pergunta

In [21]:
def responder_pergunta(pergunta, modelo_word2vec, mapa_chunks, top_n=1):
    # ... (Seu código existente para pré-processar a pergunta e extrair entidades da pergunta)
    tokens_pergunta_limpos = preprocessar_texto(pergunta)
    vetor_pergunta = obter_vetor_medio(tokens_pergunta_limpos, modelo_word2vec)

    if vetor_pergunta is None:
        return "Desculpe, não consegui entender sua pergunta."

    # Processa a pergunta com SpaCy para extração de entidades
    pergunta_doc = nlp(pergunta)
    per_pergunta = {normalizar_entidade(ent.text) for ent in pergunta_doc.ents if ent.label_ == "PER" and len(normalizar_entidade(ent.text).split()) >= 2}
    loc_pergunta = {normalizar_entidade(ent.text) for ent in pergunta_doc.ents if ent.label_ == "LOC" and len(normalizar_entidade(ent.text).split()) >= 2}


    resultados_chunks = []
    for chunk_id, dados_chunk in mapa_chunks.items():
        chunk_texto = dados_chunk['texto_original']
        vetor_chunk = dados_chunk['vetor_chunk']

        # --- PARTE A SER ALTERADA/ADICIONADA: Conversão para set e acesso às entidades ---
        # Certifique-se de que 'entidades_per' e 'entidades_loc' estejam armazenadas em mapa_chunks
        entidades_per_chunk = set(dados_chunk['entidades_per'])
        entidades_loc_chunk = set(dados_chunk['entidades_loc'])
        # --- FIM DA PARTE A SER ALTERADA/ADICIONADA ---

        # Calcula a similaridade de cosseno
        if vetor_chunk is None:
            continue

        similaridade = cosine_similarity([vetor_pergunta], [vetor_chunk])[0][0]

        # --- PARTE A SER ALTERADA: Ajuste de Peso para Pessoas (PER) ---
        bonus_per = 0
        if per_pergunta:
            matches_per = len(per_pergunta.intersection(entidades_per_chunk))
            if matches_per > 0:
                # Ajuste este peso (0.08 é um valor de exemplo, você pode testar outros)
                bonus_per = 0.08 * matches_per
        # --- FIM DA PARTE A SER ALTERADA ---

        # --- PARTE A SER ADICIONADA: Ajuste de Peso para Locais (LOC) ---
        bonus_loc = 0
        if loc_pergunta:
            matches_loc = len(loc_pergunta.intersection(entidades_loc_chunk))
            if matches_loc > 0:
                # Ajuste este peso (0.05 é um valor de exemplo, você pode testar outros)
                bonus_loc = 0.05 * matches_loc
        # --- FIM DA PARTE A SER ADICIONADA ---

        # --- PARTE A SER ALTERADA: Cálculo do Score Ajustado Final ---
        score_ajustado = similaridade + bonus_per + bonus_loc
        # --- FIM DA PARTE A SER ALTERADA ---

        resultados_chunks.append({
            'chunk_id': chunk_id,
            'score_original': similaridade,
            'score_ajustado': score_ajustado,
            'texto': chunk_texto,
            # --- PARTE A SER ALTERADA: Converter de volta para lista para exibição ---
            'entidades_per': list(entidades_per_chunk),
            'entidades_loc': list(entidades_loc_chunk)
            # --- FIM DA PARTE A SER ALTERADA ---
        })

    # ... (Seu código existente para ordenar e retornar a melhor resposta)
    resultados_chunks.sort(key=lambda x: x['score_ajustado'], reverse=True)

    if resultados_chunks:
        melhor_resposta = resultados_chunks[0]

        resposta_formatada = (
            f"🔹 Score (ajustado): {melhor_resposta['score_ajustado']:.4f}\n"
            f"{melhor_resposta['texto']}\n"
            f"   → Pessoas: {', '.join(melhor_resposta['entidades_per']) if melhor_resposta['entidades_per'] else '—'}\n"
            f"   → Locais: {', '.join(melhor_resposta['entidades_loc']) if melhor_resposta['entidades_loc'] else '—'}"
        )
        return resposta_formatada
    else:
        return "Desculpe, não encontrei uma resposta relevante."

## Execução

In [23]:
pergunta_usuario = input("Digite sua pergunta sobre o livro O Guarani:\n> ")
responder_pergunta(pergunta_usuario, model_w2v, mapa_chunks, top_n=1)

Digite sua pergunta sobre o livro O Guarani:
> Como é descrita a natureza brasileira ao redor da casa de Dom Antônio?


NameError: name 'obter_vetor_medio' is not defined